In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [104]:
!pip install paddleocr paddlepaddle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.7/193.7 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from langdetect import detect
import easyocr

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
class MultiTaskIndicBERT(nn.Module):
    def __init__(self, encoder, hate_classes, emotion_classes, sentiment_classes):
        super().__init__()

        self.encoder = encoder
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.3)

        self.hate_head = nn.Linear(hidden_size, hate_classes)
        self.emotion_head = nn.Linear(hidden_size, emotion_classes)
        self.sentiment_head = nn.Linear(hidden_size, sentiment_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = outputs.last_hidden_state[:,0]   # CLS

        pooled = self.dropout(pooled)

        hate_logits = self.hate_head(pooled)
        emotion_logits = self.emotion_head(pooled)
        sentiment_logits = self.sentiment_head(pooled)

        return hate_logits, emotion_logits, sentiment_logits

In [14]:
hindi_path = "/content/drive/MyDrive/EmiHate/Models/indicbert_hin"

hin_tokenizer = AutoTokenizer.from_pretrained(hindi_path)
hin_encoder = AutoModel.from_pretrained(hindi_path)

hin_model = MultiTaskIndicBERT(hin_encoder, 3, 5, 3)
hin_model.load_state_dict(torch.load(hindi_path + "/model_heads.pt"))
hin_model.to(device)
hin_model.eval()

Loading weights:   0%|          | 0/25 [00:02<?, ?it/s]

MultiTaskIndicBERT(
  (encoder): AlbertModel(
    (embeddings): AlbertEmbeddings(
      (word_embeddings): Embedding(200000, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): AlbertTransformer(
      (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
      (albert_layer_groups): ModuleList(
        (0): AlbertLayerGroup(
          (albert_layers): ModuleList(
            (0): AlbertLayer(
              (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (attention): AlbertAttention(
                (attention_dropout): Dropout(p=0, inplace=False)
                (output_dropout): Dropout(p=0, inplace=False)
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Lin

In [9]:
telugu_path = "/content/drive/MyDrive/EmiHate/Models/indicbert_telugu"

tel_tokenizer = AutoTokenizer.from_pretrained(telugu_path)
tel_encoder = AutoModel.from_pretrained(telugu_path)

tel_model = MultiTaskIndicBERT(tel_encoder, 3, 5, 3)
tel_model.load_state_dict(torch.load(telugu_path + "/model_heads.pt"))
tel_model.to(device)
tel_model.eval()

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

MultiTaskIndicBERT(
  (encoder): AlbertModel(
    (embeddings): AlbertEmbeddings(
      (word_embeddings): Embedding(200000, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): AlbertTransformer(
      (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
      (albert_layer_groups): ModuleList(
        (0): AlbertLayerGroup(
          (albert_layers): ModuleList(
            (0): AlbertLayer(
              (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (attention): AlbertAttention(
                (attention_dropout): Dropout(p=0, inplace=False)
                (output_dropout): Dropout(p=0, inplace=False)
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Lin

In [11]:
english_path = "/content/drive/MyDrive/EmiHate/Models/Roberta_engl"

eng_tokenizer = AutoTokenizer.from_pretrained(english_path)
eng_encoder = AutoModel.from_pretrained(english_path)

eng_model = MultiTaskIndicBERT(eng_encoder, 3, 5, 3)
eng_model.load_state_dict(torch.load(english_path + "/model_heads.pt"))
eng_model.to(device)
eng_model.eval()

Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

MultiTaskIndicBERT(
  (encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

In [78]:
import easyocr

print("🔄 Initializing OCR models...")

reader_en = easyocr.Reader(['en'])
reader_hi = easyocr.Reader(['hi', 'en'])   # Hindi works better with en
reader_te = easyocr.Reader(['te'])         # IMPORTANT: only 'te'

ocr_models = {
    "en": reader_en,
    "hi": reader_hi,
    "te": reader_te
}

print("✅ All OCR models loaded")

🔄 Initializing OCR models...
✅ All OCR models loaded


In [89]:
import cv2

def extract_text(image_path):

    try:
        # Load image
        img = cv2.imread(image_path)

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Apply threshold (improves text visibility)
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

        # Try Telugu OCR FIRST
        result_te = ocr_models["te"].readtext(thresh)

        text_te = " ".join([r[1] for r in result_te]).strip()

        # If Telugu text detected → return
        if len(text_te) > 5:
            return text_te

        # Else fallback to Hindi/English
        result_hi = ocr_models["hi"].readtext(thresh)
        text_hi = " ".join([r[1] for r in result_hi]).strip()

        if len(text_hi) > 5:
            return text_hi

        result_en = ocr_models["en"].readtext(thresh)
        text_en = " ".join([r[1] for r in result_en]).strip()

        return text_en

    except Exception as e:
        print("❌ OCR Error:", e)
        return ""

In [90]:
def detect_lang(text):

    if any('\u0C00' <= ch <= '\u0C7F' for ch in text):
        return "te"

    elif any('\u0900' <= ch <= '\u097F' for ch in text):
        return "hi"

    else:
        return "en"

In [92]:
def is_valid_text(text):
    return len(text) > 5 and not any(char.isdigit() for char in text[:5])

In [93]:
hate_map = {0:"Hate",1:"Offensive",2:"Neutral"}

emotion_map = {
    0:"anger",
    1:"hate",
    2:"sadness",
    3:"fear",
    4:"neutral"
}

sentiment_map = {0:"Negative",1:"Neutral",2:"Positive"}

In [94]:
def predict_text(text):

    lang = detect_lang(text)

    if lang == "en":
        tokenizer, model = eng_tokenizer, eng_model
    elif lang == "hi":
        tokenizer, model = hin_tokenizer, hin_model
    elif lang == "te":
        tokenizer, model = tel_tokenizer, tel_model
    else:
        print("Unsupported language")
        return

    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        h, e, s = model(input_ids, attention_mask)

    hate_probs = F.softmax(h, dim=1)[0].cpu().numpy()
    emotion_probs = F.softmax(e, dim=1)[0].cpu().numpy()
    sentiment_probs = F.softmax(s, dim=1)[0].cpu().numpy()

    print("\n🌐 Language:", lang)
    print("📝 Text:", text)

    # ---------------- HATE ----------------
    print("\n🚨 Hate Detection:")
    for i,p in enumerate(hate_probs):
        print(f"{hate_map[i]}: {p*100:.2f}%")

    final_hate = hate_map[hate_probs.argmax()]
    print("➡ FINAL:", final_hate)

    print("-"*50)

    # ---------------- EMOTION (TOP 2 ONLY) ----------------
    print("🧠 Top Emotions:")

    sorted_idx = emotion_probs.argsort()[::-1]
    top2 = sorted_idx[:2]

    for idx in top2:
        print(f"{emotion_map[idx]}: {emotion_probs[idx]*100:.2f}%")

    # confidence check
    final_idx = sorted_idx[0]
    final_prob = emotion_probs[final_idx]

    if final_prob < 0.35:
        final_emotion = emotion_map[final_idx]
    else:
        final_emotion = emotion_map[final_idx]

    print("➡ FINAL EMOTION:", final_emotion)

    print("-"*50)

    # ---------------- SENTIMENT ----------------
    print("📊 Sentiment:")
    for i,p in enumerate(sentiment_probs):
        print(f"{sentiment_map[i]}: {p*100:.2f}%")

    final_sentiment = sentiment_map[sentiment_probs.argmax()]
    print("➡ FINAL SENTIMENT:", final_sentiment)

    print("="*60)

In [95]:
def run_pipeline(image_path):

    text = extract_text(image_path)

    print("🔍 OCR TEXT:\n", text)

    print("\n========================\n")

    predict_text(text)

In [96]:
predict_text("तुम बिल्कुल बेकार हो 😡")


🌐 Language: hi
📝 Text: तुम बिल्कुल बेकार हो 😡

🚨 Hate Detection:
Hate: 31.27%
Offensive: 33.14%
Neutral: 35.58%
➡ FINAL: Neutral
--------------------------------------------------
🧠 Top Emotions:
fear: 20.82%
anger: 20.76%
➡ FINAL EMOTION: fear
--------------------------------------------------
📊 Sentiment:
Negative: 33.17%
Neutral: 35.51%
Positive: 31.31%
➡ FINAL SENTIMENT: Neutral


In [97]:
predict_text("నువ్వు పనికిరాని వాడివి 😡")


🌐 Language: te
📝 Text: నువ్వు పనికిరాని వాడివి 😡

🚨 Hate Detection:
Hate: 34.26%
Offensive: 35.43%
Neutral: 30.31%
➡ FINAL: Offensive
--------------------------------------------------
🧠 Top Emotions:
anger: 22.89%
fear: 20.25%
➡ FINAL EMOTION: anger
--------------------------------------------------
📊 Sentiment:
Negative: 33.84%
Neutral: 33.14%
Positive: 33.02%
➡ FINAL SENTIMENT: Negative


In [98]:
predict_text("You are useless and stupid 😡")


🌐 Language: en
📝 Text: You are useless and stupid 😡

🚨 Hate Detection:
Hate: 27.69%
Offensive: 48.01%
Neutral: 24.30%
➡ FINAL: Offensive
--------------------------------------------------
🧠 Top Emotions:
sadness: 26.55%
anger: 23.94%
➡ FINAL EMOTION: sadness
--------------------------------------------------
📊 Sentiment:
Negative: 53.69%
Neutral: 34.18%
Positive: 12.13%
➡ FINAL SENTIMENT: Negative


In [99]:
predict_text("/content/drive/MyDrive/EmiHate/testing_images/test_english.png")


🌐 Language: en
📝 Text: /content/drive/MyDrive/EmiHate/testing_images/test_english.png

🚨 Hate Detection:
Hate: 20.83%
Offensive: 32.27%
Neutral: 46.89%
➡ FINAL: Neutral
--------------------------------------------------
🧠 Top Emotions:
sadness: 25.69%
neutral: 21.92%
➡ FINAL EMOTION: sadness
--------------------------------------------------
📊 Sentiment:
Negative: 39.59%
Neutral: 38.58%
Positive: 21.83%
➡ FINAL SENTIMENT: Negative


In [100]:
run_pipeline("/content/drive/MyDrive/EmiHate/testing_images/download.jpg")

🔍 OCR TEXT:
 నేను ద్వేషిస్తున్నాను అంటున కానీ నా దయం మాతం ఇంకా నిన్నే ప్రేమిస్తోంది



🌐 Language: te
📝 Text: నేను ద్వేషిస్తున్నాను అంటున కానీ నా దయం మాతం ఇంకా నిన్నే ప్రేమిస్తోంది

🚨 Hate Detection:
Hate: 34.25%
Offensive: 35.29%
Neutral: 30.46%
➡ FINAL: Offensive
--------------------------------------------------
🧠 Top Emotions:
anger: 22.82%
fear: 20.16%
➡ FINAL EMOTION: anger
--------------------------------------------------
📊 Sentiment:
Negative: 33.95%
Neutral: 33.15%
Positive: 32.90%
➡ FINAL SENTIMENT: Negative


In [101]:
run_pipeline("/content/drive/MyDrive/EmiHate/testing_images/telug.png")

🔍 OCR TEXT:
 పది అడుగులు దూరం పెట్టాలనుకునే వారికి నువ్వే వందడుగుల దూరం ఉండి పనిచేసుకో



🌐 Language: te
📝 Text: పది అడుగులు దూరం పెట్టాలనుకునే వారికి నువ్వే వందడుగుల దూరం ఉండి పనిచేసుకో

🚨 Hate Detection:
Hate: 34.33%
Offensive: 35.30%
Neutral: 30.37%
➡ FINAL: Offensive
--------------------------------------------------
🧠 Top Emotions:
anger: 22.81%
fear: 20.37%
➡ FINAL EMOTION: anger
--------------------------------------------------
📊 Sentiment:
Negative: 33.86%
Neutral: 33.20%
Positive: 32.94%
➡ FINAL SENTIMENT: Negative


In [102]:
run_pipeline("/content/drive/MyDrive/EmiHate/testing_images/telugu-1.jpg")

🔍 OCR TEXT:
 ;4ఓబడ* పలాగ5 5ే6ం 50ై ౌ0ఠఢఓహా సేగాడ గాదాా? 6 ూసడన2 412 78[ ఘ 07511_ పఠ?~ {5స"#ేఊ  నేసొన4 ోఓ~ఐస ఖఫ& గం 'ఓన61సేగ 0ఫ~= ఐ] 143 @6' {వర్డ1న'' 18.% 1:. %9గ క:'8ః 0పనాగడగా} స#1: [హూటకలకగాయ ` #సమంగేట 721 7312సుే6ౢ ః 114 హ1ఇ 1న5గ



🌐 Language: te
📝 Text: ;4ఓబడ* పలాగ5 5ే6ం 50ై ౌ0ఠఢఓహా సేగాడ గాదాా? 6 ూసడన2 412 78[ ఘ 07511_ పఠ?~ {5స"#ేఊ  నేసొన4 ోఓ~ఐస ఖఫ& గం 'ఓన61సేగ 0ఫ~= ఐ] 143 @6' {వర్డ1న'' 18.% 1:. %9గ క:'8ః 0పనాగడగా} స#1: [హూటకలకగాయ ` #సమంగేట 721 7312సుే6ౢ ః 114 హ1ఇ 1న5గ

🚨 Hate Detection:
Hate: 34.13%
Offensive: 35.30%
Neutral: 30.57%
➡ FINAL: Offensive
--------------------------------------------------
🧠 Top Emotions:
anger: 22.94%
fear: 20.24%
➡ FINAL EMOTION: anger
--------------------------------------------------
📊 Sentiment:
Negative: 34.03%
Neutral: 33.26%
Positive: 32.71%
➡ FINAL SENTIMENT: Negative


In [103]:
run_pipeline("/content/drive/MyDrive/EmiHate/testing_images/hindi.jpg")

🔍 OCR TEXT:
 #7]0ుఫ`` 37ఘ*7711 @1 #ఈఘ ెగ% గై థ1?గెం్ధ? ఃఅఖన 37 'ొఛ@ ఖ ఈూక@ & గ్్ ె/ేఇ డ1థ ేఎఉ సే



🌐 Language: te
📝 Text: #7]0ుఫ`` 37ఘ*7711 @1 #ఈఘ ెగ% గై థ1?గెం్ధ? ఃఅఖన 37 'ొఛ@ ఖ ఈూక@ & గ్్ ె/ేఇ డ1థ ేఎఉ సే

🚨 Hate Detection:
Hate: 34.00%
Offensive: 35.40%
Neutral: 30.60%
➡ FINAL: Offensive
--------------------------------------------------
🧠 Top Emotions:
anger: 22.84%
fear: 20.16%
➡ FINAL EMOTION: anger
--------------------------------------------------
📊 Sentiment:
Negative: 33.93%
Neutral: 33.15%
Positive: 32.92%
➡ FINAL SENTIMENT: Negative
